# 04 — Visualisations (v2)
**Input:** `data/processed/upl_hsr_wide.csv` | **Output:** `output/figures/` — 8 PNGs

In [5]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = 'Cabin'
matplotlib.rcParams['figure.dpi']  = 200

WIDE_FILE  = '../data/processed/upl_hsr_wide.csv'
OUTPUT_DIR = '../output/figures/'
org_LOGO  = '../assets/org_logo.png'  
FIG_W, FIG_H = 14.0, 9.0
DPI = 200
print('Setup complete.')


Setup complete.


In [7]:
wide = pd.read_csv(WIDE_FILE)

POS_ORDER  = ['DEF', 'MID', 'FWD']
POS_LABELS = {'DEF': 'Defenders', 'MID': 'Midfielders', 'FWD': 'Forwards'}

club_summary = (
    wide.groupby('club_for')
    .agg(n=('decay_abs','count'),
         mean_h1=('h1_rate','mean'),
         mean_h2=('h2_rate','mean'),
         mean_decay=('decay_abs','mean'))
    .reset_index()
    .sort_values('mean_decay', ascending=True)
    .reset_index(drop=True)
)

print(f'Rows: {len(wide):,} | Clubs: {wide["club_for"].nunique()}')
print(club_summary[['club_for','n','mean_decay']].to_string(index=False))


Rows: 3,441 | Clubs: 16
club_for   n  mean_decay
 Club-01 217    1.148197
 Club-10 253    1.782163
 Club-13 218    1.887113
 Club-11 171    1.961172
 Club-12 132    2.191836
 Club-02 247    2.224177
 Club-08  77    2.366819
 Club-07 237    2.524333
 Club-15 261    2.799041
 Club-09 270    2.808922
 Club-14 257    2.905545
 Club-16 224    3.052637
 Club-05 266    3.183080
 Club-03  76    3.572526
 Club-06 278    3.696812
 Club-04 257    4.343346


In [8]:
THEMES = {
    'org': {
        'name':'org', 'bg':'#FFFFFF', 'panel_bg':'#F0F5FC',
        'grid':'#D8E4F2', 'accent1':'#2b499f', 'accent2':'#D95E00',
        'accent3':'#047ac2', 'text_dark':'#0D1B3E', 'text_mid':'#3A5080',
        'text_light':'#8A9EBB', 'spine':'#C8D8EE',
        'decay_high':'#C0392B', 'decay_low':'#1A7A4A', 'decay_mid':'#6B82A8',
        'h1_color':'#2b499f', 'h2_color':'#D95E00',
    },
    'portfolio': {
        'name':'portfolio', 'bg':'#0D1117', 'panel_bg':'#161B25',
        'grid':'#1E2535', 'accent1':'#00D4FF', 'accent2':'#FF6B35',
        'accent3':'#7FBBFF', 'text_dark':'#E8EDF8', 'text_mid':'#8A9BBF',
        'text_light':'#3D4A60', 'spine':'#1E2535',
        'decay_high':'#FF4D6D', 'decay_low':'#00E5A0', 'decay_mid':'#3D5070',
        'h1_color':'#00D4FF', 'h2_color':'#FF6B35',
    }
}
print('Themes ready.')


Themes ready.


In [ ]:
import os

def add_top_bar(fig, t):
    tb = fig.add_axes([0.03, 0.974, 0.94, 0.007])
    if t['name'] == 'org':
        tb.imshow(np.linspace(0,1,256).reshape(1,256), aspect='auto',
                  cmap=mcolors.LinearSegmentedColormap.from_list('g',[t['accent1'],t['accent3']]))
    else:
        tb.set_facecolor(t['accent1'])
    tb.set_axis_off()

def add_branding(fig, t):
    if org_LOGO:
        try:
            from matplotlib.image import imread
            logo = imread(org_LOGO)
            al = fig.add_axes([0.89, 0.905, 0.065, 0.055])
            al.imshow(logo); al.set_axis_off(); return
        except Exception:
            pass
    fig.text(0.955, 0.950, 'org RST', fontsize=9, fontweight='bold',
             color=t['accent1'], ha='right', va='top')

def add_footer(fig, t, org_version):
    src   = 'Source: GPS  ·  UPL  ·  2024/25 Season'
    right = 'org  ·  RST' if org_version else 'UPL GPS Analysis  ·  2024/25'
    fig.text(0.04, 0.022, src,   fontsize=8, color=t['text_light'])
    fig.text(0.96, 0.022, right, fontsize=8, color=t['text_light'], ha='right')

def make_figure(t, org_version, has_panel):
    fig    = plt.figure(figsize=(FIG_W, FIG_H), facecolor=t['bg'], dpi=DPI)
    ax_hdr = fig.add_axes([0.03, 0.875, 0.94, 0.092])
    ax_hdr.set_facecolor(t['bg']); ax_hdr.set_axis_off()
    if has_panel:
        ax_main = fig.add_axes([0.04, 0.12, 0.60, 0.725])
        ax_ann  = fig.add_axes([0.67, 0.12, 0.30, 0.725])
    else:
        ax_main = fig.add_axes([0.06, 0.12, 0.90, 0.725])
        ax_ann  = None
    ax_main.set_facecolor(t['bg'])
    add_top_bar(fig, t)
    if org_version: add_branding(fig, t)
    add_footer(fig, t, org_version)
    return fig, ax_hdr, ax_main, ax_ann

def style_header(ax, title, subtitle, t):
    ax.text(0.5, 0.94, title, fontsize=17, fontweight='bold', color=t['text_dark'],
            ha='center', va='top', transform=ax.transAxes)
    ax.text(0.5, 0.28, subtitle, fontsize=9.5, color=t['text_mid'],
            ha='center', va='top', transform=ax.transAxes)

def add_panel(ax, t, blocks):
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.add_patch(FancyBboxPatch((0,0),1,1, boxstyle='round,pad=0.015', lw=0.8,
        edgecolor=t['grid'], facecolor=t['panel_bg'], zorder=0,
        clip_on=False, transform=ax.transAxes))
    ax.add_patch(Rectangle((0,0),0.035,1, lw=0, facecolor=t['accent1'],
        zorder=1, clip_on=False, transform=ax.transAxes))
    T=ax.transAxes; LH=0.042; BG=0.028; PG=0.010; SEP=0.022

    def hline(y):
        ax.plot([0.07,0.96],[y,y], color=t['grid'], lw=0.7, transform=T, clip_on=False)

    y = 0.96
    for idx, block in enumerate(blocks):
        lc = block.get('label_color', t['accent1'])
        ax.text(0.07, y, block['label'], color=lc, fontsize=8.2, fontweight='bold',
                va='top', ha='left', transform=T, zorder=5)
        y -= BG
        for pi, para in enumerate(block['lines']):
            if pi > 0: y -= PG
            bold = (pi==0 and block.get('bold_first', False))
            for line in para:
                ax.text(0.07, y, line,
                        color=t['text_dark'] if bold else t['text_mid'],
                        fontsize=9.0 if bold else 8.6,
                        fontweight='semibold' if bold else 'normal',
                        va='top', ha='left', transform=T, zorder=5)
                y -= LH
        y -= SEP
        if idx < len(blocks)-1:
            hline(y + SEP*0.5)
    ax.set_axis_off()

def save_chart(fig, num, theme_name):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path = f'{OUTPUT_DIR}chart{num}_{theme_name}.png'
    fig.savefig(path, dpi=DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f'Saved: {path}')

print('Helpers ready.')


Helpers ready.


In [10]:
def chart1(theme_name):
    t    = THEMES[theme_name]
    org = (theme_name == 'org')
    mean_h1   = wide['h1_rate'].mean()
    mean_h2   = wide['h2_rate'].mean()
    decay_abs = mean_h1 - mean_h2
    decay_pct = decay_abs / mean_h1 * 100

    fig, ax_hdr, ax, ax_ann = make_figure(t, org, has_panel=org)
    style_header(ax_hdr,
        'Players Lose a Quarter of Their High-Speed Running in the Second Half',
        f'HSR Rate (m/min  ·  speed zones ≥18 km/h)  ·  1st Half vs 2nd Half  ·  All 16 UPL Clubs  ·  n = {len(wide):,} player-match observations',
        t)

    np.random.seed(42)
    sample = wide.sample(min(1400, len(wide)), random_state=42)
    for _, row in sample.iterrows():
        clr = t['decay_high'] if row['decay_abs'] > 0 else t['decay_low']
        ax.plot([0,1],[row['h1_rate'],row['h2_rate']],
                color=clr, alpha=0.025 if org else 0.03, linewidth=0.7, zorder=2)

    jw = 0.055
    jx1 = np.random.uniform(-jw, jw, len(sample))
    jx2 = np.random.uniform(-jw, jw, len(sample))
    ax.scatter(jx1,     sample['h1_rate'], color=t['h1_color'], alpha=0.06, s=7, zorder=3)
    ax.scatter(1 + jx2, sample['h2_rate'], color=t['h2_color'], alpha=0.06, s=7, zorder=3)

    ax.plot([0,1],[mean_h1,mean_h2], color=t['text_dark'], linewidth=3.2, zorder=7, solid_capstyle='round')
    ax.scatter(0, mean_h1, s=190, color=t['h1_color'], zorder=9, edgecolors=t['bg'], linewidths=2.5)
    ax.scatter(1, mean_h2, s=190, color=t['h2_color'], zorder=9, edgecolors=t['bg'], linewidths=2.5)
    ax.text(0,  mean_h1+0.7,  f'{mean_h1:.2f}', color=t['h1_color'], fontsize=16, fontweight='bold', ha='center', va='bottom', zorder=10)
    ax.text(1,  mean_h2-0.7,  f'{mean_h2:.2f}', color=t['h2_color'], fontsize=16, fontweight='bold', ha='center', va='top',    zorder=10)

    mid_y = (mean_h1 + mean_h2) / 2
    ax.annotate(f'−{decay_abs:.2f} m/min\n−{decay_pct:.1f}%',
        xy=(0.5, mid_y), xytext=(0.5, mid_y+2.8),
        color=t['text_dark'], fontsize=12, fontweight='bold', ha='center', va='bottom',
        arrowprops=dict(arrowstyle='->', color=t['text_mid'], lw=1.6), zorder=10)

    ax.set_xlim(-0.30, 1.30)
    ax.set_ylim(1.0, 35)
    ax.set_xticks([0,1])
    ax.set_xticklabels(['1st Half','2nd Half'], fontsize=14, fontweight='bold', color=t['text_dark'])
    ax.set_ylabel('HSR Rate (m/min)', color=t['text_mid'], fontsize=11, labelpad=10)
    ax.tick_params(axis='y', colors=t['text_mid'], labelsize=10)
    ax.tick_params(axis='x', length=0)
    ax.grid(axis='y', color=t['grid'], linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    for sp in ['top','right','bottom']: ax.spines[sp].set_visible(False)
    ax.spines['left'].set_color(t['spine'])
    for xi in [0,1]: ax.axvline(xi, color=t['grid'], linewidth=0.8, zorder=1)
    for xi, lbl, ci in [(0,'1ST HALF',t['h1_color']),(1,'2ND HALF',t['h2_color'])]:
        ax.text(xi, 33.5, lbl, color=ci, fontsize=9, fontweight='bold', ha='center', va='top', alpha=0.7)

    if org and ax_ann is not None:
        add_panel(ax_ann, t, [
            {'label':'WHAT THIS TELLS COACHES', 'label_color':t['accent1'], 'bold_first':True,
             'lines': [
                [f'Players average {mean_h1:.2f} m/min of high-speed running',
                 f'in the first half, dropping to {mean_h2:.2f} in the second',
                 f'— a {decay_pct:.1f}% decline.'],
                ['This pattern holds across all 16 UPL clubs.', 'It is not an outlier effect.'],
             ]},
            {'label':'ACTION POINT', 'label_color':t['accent2'], 'bold_first':False,
             'lines': [
                ['Prioritise second-half conditioning.',
                 'Repeat-sprint and aerobic capacity work should',
                 'form the core of pre-season fitness blocks.'],
             ]},
            {'label':'HOW TO READ THIS CHART', 'label_color':t['text_mid'], 'bold_first':False,
             'lines': [
                ['Each faint line = one player-match observation.',
                 'Bold line = league mean.',
                 'Red lines = decay  ·  Green lines = improvement.'],
             ]},
        ])
    save_chart(fig, 1, theme_name)

# chart1('org')
chart1('portfolio')


Saved: ../output/figures/chart1_portfolio.png


In [11]:
def chart2(theme_name):
    t    = THEMES[theme_name]
    org = (theme_name == 'org')

    pos_agg = (
        wide.groupby('position')
        .agg(mean_h1=('h1_rate','mean'), mean_h2=('h2_rate','mean'),
             mean_decay=('decay_abs','mean'), n=('decay_abs','count'))
        .loc[POS_ORDER].reset_index()
    )
    pos_agg['decay_pct'] = pos_agg['mean_decay'] / pos_agg['mean_h1'] * 100

    pos_colors = {'DEF':t['accent1'], 'MID':t['accent3'], 'FWD':t['accent2']}

    fig, ax_hdr, ax, ax_ann = make_figure(t, org, has_panel=org)
    style_header(ax_hdr,
        'Forwards Fade Fastest — Defenders Hold Their Pace Best',
        'Mean HSR Rate by Position  ·  1st Half → 2nd Half  ·  2024/25 UPL Season',
        t)

    ax.set_facecolor(t['bg'])

    for _, row in pos_agg.iterrows():
        pos   = row['position']
        h1, h2 = row['mean_h1'], row['mean_h2']
        decay  = row['mean_decay']
        pct    = row['decay_pct']
        color  = pos_colors[pos]

        ax.plot([0,1],[h1,h2], color=color, linewidth=4.0, zorder=5, solid_capstyle='round')
        ax.scatter(0, h1, s=230, color=color, zorder=7, edgecolors=t['bg'], linewidths=2.5)
        ax.scatter(1, h2, s=230, color=color, zorder=7, edgecolors=t['bg'], linewidths=2.5)

        # H1 label — left of column
        ax.text(-0.06, h1, f'{h1:.2f}', color=color, fontsize=13, fontweight='bold',
                ha='right', va='center', zorder=8)
        # H2 label + position name — right of column
        ax.text(1.06, h2, f'{h2:.2f}  {POS_LABELS[pos]}', color=color, fontsize=13,
                fontweight='bold', ha='left', va='center', zorder=8)

        # Decay callout — offset to avoid overlap
        y_mid    = (h1 + h2) / 2
        offsets  = {'DEF': +0.50, 'MID': -0.55, 'FWD': +0.50}
        y_offset = offsets[pos]
        ax.text(0.5, y_mid + y_offset,
                f'−{decay:.2f} m/min  ({pct:.0f}%)',
                color=color, fontsize=10, fontweight='bold', ha='center', va='center', zorder=8,
                bbox=dict(boxstyle='round,pad=0.28', facecolor=t['bg'],
                          edgecolor=color, linewidth=0.9, alpha=0.92))

    # Column headers
    y_top = pos_agg['mean_h1'].max() + 1.6
    ax.text(0, y_top, '1ST HALF', color=t['text_dark'], fontsize=13,
            fontweight='bold', ha='center', va='bottom')
    ax.text(1, y_top, '2ND HALF', color=t['text_dark'], fontsize=13,
            fontweight='bold', ha='center', va='bottom')
    for xi in [0,1]:
        ax.axvline(xi, color=t['grid'], linewidth=1.0, zorder=1, alpha=0.7)

    # Axes
    y_min = pos_agg['mean_h2'].min() - 1.4
    y_max = y_top + 0.5
    ax.set_xlim(-0.55, 1.85)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks([])
    ax.set_ylabel('Mean HSR Rate (m/min)', color=t['text_mid'], fontsize=11, labelpad=10)
    ax.tick_params(axis='y', colors=t['text_mid'], labelsize=10)
    ax.grid(axis='y', color=t['grid'], linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    for sp in ['top','right','bottom']: ax.spines[sp].set_visible(False)
    ax.spines['left'].set_color(t['spine'])

    # Legend
    handles = [Line2D([0],[0], color=pos_colors[p], linewidth=4, label=POS_LABELS[p]) for p in POS_ORDER]
    ax.legend(handles=handles, loc='lower left', frameon=True,
              facecolor=t['panel_bg'], edgecolor=t['grid'],
              fontsize=10, labelcolor=t['text_mid'])

    if org and ax_ann is not None:
        fwd_pct = pos_agg.loc[pos_agg['position']=='FWD','decay_pct'].values[0]
        def_pct = pos_agg.loc[pos_agg['position']=='DEF','decay_pct'].values[0]
        fwd_abs = pos_agg.loc[pos_agg['position']=='FWD','mean_decay'].values[0]
        def_abs = pos_agg.loc[pos_agg['position']=='DEF','mean_decay'].values[0]
        add_panel(ax_ann, t, [
            {'label':'WHAT THIS TELLS COACHES', 'label_color':t['accent1'], 'bold_first':True,
             'lines': [
                [f'Forwards lose {fwd_abs:.2f} m/min ({fwd_pct:.0f}%) of their',
                 f'high-speed pace by the second half — vs only',
                 f'{def_abs:.2f} m/min ({def_pct:.0f}%) for defenders.'],
                ['The positional gap is consistent across all', 'match days in the season.'],
             ]},
            {'label':'ACTION POINT', 'label_color':t['accent2'], 'bold_first':False,
             'lines': [
                ['Design position-specific fitness blocks.',
                 'Forwards need repeat-sprint protocols.',
                 'Consider earlier attacking substitutions',
                 'to preserve high-speed threat in the second half.'],
             ]},
            {'label':'NOTE ON MIDFIELDERS', 'label_color':t['accent3'], 'bold_first':False,
             'lines': [
                ['Midfielders sit between the two positions',
                 'in fatigue profile — closer to defenders.'],
             ]},
        ])
    save_chart(fig, 2, theme_name)

# chart2('org')
chart2('portfolio')


Saved: ../output/figures/chart2_portfolio.png


In [12]:
def chart3(theme_name):
    t    = THEMES[theme_name]
    org = (theme_name == 'org')

    df      = club_summary.copy()
    n_clubs = len(df)
    ranks   = df['mean_decay'].rank(ascending=False)
    colors  = []
    for i in df.index:
        r = ranks[i]
        if r <= 4:            colors.append(t['decay_high'])
        elif r > n_clubs - 4: colors.append(t['decay_low'])
        else:                 colors.append(t['decay_mid'])
    df['color'] = colors

    top_club  = df.iloc[-1]['club_for']
    top_decay = df.iloc[-1]['mean_decay']
    bot_club  = df.iloc[0]['club_for']
    bot_decay = df.iloc[0]['mean_decay']
    ratio     = top_decay / bot_decay

    fig, ax_hdr, ax, ax_ann = make_figure(t, org, has_panel=org)
    style_header(ax_hdr,
        f'{top_club} Players Lose {ratio:.1f}× More Pace Than {bot_club}',
        'Mean HSR Decay (m/min)  ·  1st Half minus 2nd Half  ·  All 16 UPL Clubs  ·  2024/25 Season',
        t)

    ax.set_facecolor(t['bg'])
    y_pos = np.arange(n_clubs)
    bars  = ax.barh(y_pos, df['mean_decay'].values,
                    color=df['color'].values, height=0.64,
                    edgecolor=t['bg'], linewidth=0.5, zorder=3)

    max_decay = df['mean_decay'].max()
    for i, (val, clr) in enumerate(zip(df['mean_decay'].values, df['color'].values)):
        ax.text(val + max_decay*0.016, i, f'{val:.2f}',
                color=clr, fontsize=10, fontweight='bold', va='center', ha='left', zorder=5)

    league_mean = df['mean_decay'].mean()
    ax.axvline(league_mean, color=t['text_mid'], linewidth=1.3, linestyle='--', alpha=0.65, zorder=2)
    ax.text(league_mean + max_decay*0.02, n_clubs-0.45,
            f'League avg  {league_mean:.2f}',
            color=t['text_mid'], fontsize=8.5, va='top', ha='left', alpha=0.85)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(df['club_for'].values, fontsize=10, color=t['text_dark'])
    legend_patches = [
        mpatches.Patch(color=t['decay_high'], label='Highest decay (top 4)'),
        mpatches.Patch(color=t['decay_mid'],  label='Middle clubs'),
        mpatches.Patch(color=t['decay_low'],  label='Lowest decay (bottom 4)'),
    ]
    ax.legend(handles=legend_patches, loc='lower right', frameon=True,
              facecolor=t['panel_bg'], edgecolor=t['grid'],
              fontsize=9.5, labelcolor=t['text_mid'])

    ax.set_xlim(0, max_decay * 1.32)
    ax.set_xlabel('Mean HSR Decay (m/min)', color=t['text_mid'], fontsize=11, labelpad=10)
    ax.tick_params(axis='x', colors=t['text_mid'], labelsize=10)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', color=t['grid'], linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
    ax.spines['bottom'].set_color(t['spine'])

    if org and ax_ann is not None:
        add_panel(ax_ann, t, [
            {'label':'WHAT THIS TELLS COACHES', 'label_color':t['accent1'], 'bold_first':True,
             'lines': [
                [f'{top_club} ({top_decay:.2f} m/min) decays {ratio:.1f}× more',
                 f'than {bot_club} ({bot_decay:.2f} m/min).'],
                ['Every club shows second-half decay.',
                 'The range across clubs reflects real',
                 'differences in conditioning — not just tactics.'],
             ]},
            {'label':'ACTION POINT', 'label_color':t['accent2'], 'bold_first':False,
             'lines': [
                ['Red clubs should review pre-season fitness',
                 'programming and in-season load management.',
                 'This data provides an evidence base for',
                 'targeted fitness staff conversations.'],
             ]},
            {'label':'CAUTION', 'label_color':t['text_mid'], 'bold_first':False,
             'lines': [
                ['Sample sizes vary by club.',
                 'Club-03(n=76) carries more uncertainty',
                 'than clubs with larger observations.'],
             ]},
        ])
    save_chart(fig, 3, theme_name)

# chart3('org')
chart3('portfolio')


Saved: ../output/figures/chart3_portfolio.png


In [13]:
def chart4(theme_name):
    t    = THEMES[theme_name]
    org = (theme_name == 'org')

    pos_colors = {'DEF':t['accent1'], 'MID':t['accent3'], 'FWD':t['accent2']}

    fig, ax_hdr, ax, ax_ann = make_figure(t, org, has_panel=org)
    style_header(ax_hdr,
        'The Averages Hide Wide Individual Variation in Fatigue Response',
        'Distribution of HSR Decay (m/min)  ·  Positive = Decay  ·  Negative = Improved in 2nd Half  ·  By Position',
        t)

    ax.set_facecolor(t['bg'])
    x_map = {'DEF':0, 'MID':1, 'FWD':2}

    # Shaded zones
    ax.axhspan( 0, 24, facecolor=t['decay_high'], alpha=0.04, zorder=0)
    ax.axhspan(-24, 0, facecolor=t['decay_low'],  alpha=0.04, zorder=0)

    # Zero line
    ax.axhline(0, color=t['text_mid'], linewidth=1.4, linestyle='--', alpha=0.55, zorder=4)
    ax.text(2.49, 0.5, 'No change', color=t['text_mid'], fontsize=9, va='bottom', ha='right', alpha=0.75)

    for pos in POS_ORDER:
        xi    = x_map[pos]
        data  = wide[wide['position']==pos]['decay_abs'].dropna().values
        data  = data[np.abs(data) < 22]
        color = pos_colors[pos]

        # Jittered strip
        np.random.seed(xi+7)
        idx    = np.random.choice(len(data), min(400, len(data)), replace=False)
        strip  = data[idx]
        jitter = np.random.uniform(-0.14, 0.14, len(strip))
        ax.scatter(xi+jitter, strip, color=color, alpha=0.09, s=14, zorder=2)

        # IQR box
        q25, median, q75 = np.percentile(data, [25,50,75])
        iqr = q75 - q25
        lo  = max(data.min(), q25 - 1.5*iqr)
        hi  = min(data.max(), q75 + 1.5*iqr)

        bw = 0.32
        ax.add_patch(mpatches.FancyBboxPatch(
            (xi - bw/2, q25), bw, iqr,
            boxstyle='round,pad=0.015',
            facecolor=color, alpha=0.22,
            edgecolor=color, linewidth=2.0, zorder=5))

        # Median line
        ax.plot([xi-bw/2, xi+bw/2], [median, median],
                color=color, linewidth=3.0, zorder=7, solid_capstyle='round')

        # Whiskers
        ax.vlines(xi, lo, q25, color=color, linewidth=1.5, alpha=0.55, zorder=5)
        ax.vlines(xi, q75, hi, color=color, linewidth=1.5, alpha=0.55, zorder=5)
        ax.hlines([lo, hi], xi-0.09, xi+0.09, color=color, linewidth=1.5, alpha=0.55, zorder=5)

        # Mean diamond
        mean_val = data.mean()
        ax.scatter(xi, mean_val, s=140, color=color, marker='D',
                   zorder=9, edgecolors=t['bg'], linewidths=2.0)

        # Labels below
        ax.text(xi, -19.5, POS_LABELS[pos], color=color, fontsize=13,
                fontweight='bold', ha='center', va='bottom', zorder=9)
        ax.text(xi, -21.5, f'median {median:.1f}  ·  mean {mean_val:.1f}  ·  n={len(data):,}',
                color=t['text_mid'], fontsize=8.5, ha='center', va='bottom', zorder=9)
        pct_imp = (data < 0).sum() / len(data) * 100
        ax.text(xi, -23.5, f'{pct_imp:.0f}% improved in 2nd half',
                color=t['text_light'], fontsize=8.0, ha='center', va='bottom', zorder=9)

    # Zone labels
    ax.text(-0.48, 12,  '▲ Decay zone',       color=t['decay_high'], fontsize=9, alpha=0.75, va='center', ha='left', fontweight='bold')
    ax.text(-0.48, -12, '▼ Improvement zone',  color=t['decay_low'],  fontsize=9, alpha=0.75, va='center', ha='left', fontweight='bold')

    # Legend
    legend_handles = [
        Line2D([0],[0], marker='D', color='w', markerfacecolor=t['text_mid'],
               markersize=9, label='Mean', markeredgecolor=t['bg'], markeredgewidth=1.5),
        Line2D([0],[0], color=t['text_mid'], linewidth=3, label='Median'),
        mpatches.Patch(color=t['text_mid'], alpha=0.3, label='IQR (box)'),
    ]
    ax.legend(handles=legend_handles, loc='upper right', frameon=True,
              facecolor=t['panel_bg'], edgecolor=t['grid'],
              fontsize=9.5, labelcolor=t['text_mid'])

    ax.set_xlim(-0.58, 2.58)
    ax.set_ylim(-26, 21)
    ax.set_xticks([])
    ax.set_ylabel('HSR Decay (m/min)  ·  H1 minus H2', color=t['text_mid'], fontsize=11, labelpad=10)
    ax.tick_params(axis='y', colors=t['text_mid'], labelsize=10)
    ax.grid(axis='y', color=t['grid'], linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    for sp in ['top','right','bottom']: ax.spines[sp].set_visible(False)
    ax.spines['left'].set_color(t['spine'])

    if org and ax_ann is not None:
        improved = {pos: (wide[wide['position']==pos]['decay_abs'] < 0).mean()*100 for pos in POS_ORDER}
        add_panel(ax_ann, t, [
            {'label':'WHAT THIS TELLS COACHES', 'label_color':t['accent1'], 'bold_first':True,
             'lines': [
                ['Averages hide wide individual variation.',
                 f'{improved["FWD"]:.0f}% of forwards actually run more',
                 'at high speed in the second half.'],
                ['The IQR box spans a wide range for every position',
                 '— individual players diverge sharply from the mean.'],
             ]},
            {'label':'ACTION POINT', 'label_color':t['accent2'], 'bold_first':False,
             'lines': [
                ['Use individual decay profiles to flag players',
                 '— not just position averages.',
                 'Large decay = conditioning review.',
                 'Negative decay = possible pacing strategy.'],
             ]},
            {'label':'HOW TO READ THIS CHART', 'label_color':t['text_mid'], 'bold_first':False,
             'lines': [
                ['Diamond = mean.  Horizontal line = median.',
                 'Box = middle 50% of observations (IQR).',
                 'Whiskers = 1.5× IQR.  Dots = observations.'],
             ]},
        ])
    save_chart(fig, 4, theme_name)

# chart4('org')
chart4('portfolio')


Saved: ../output/figures/chart4_portfolio.png
